In [1]:
# 05_decision_curve_and_lift.ipynb
# Actuarial utility analysis on leak-free calibrated risk scores. Produces (i) Decision
# Curve Analysis (net benefit vs threshold probability; model / treat-all / treat-none),
# (ii) Lift charts identifying the high-risk segment (cumulative lift >= 2), and
# (iii) the non-diabetic high-risk cohort used for counterfactual generation.
# Grayscale figures, dpi 600, PNG+PDF, no titles/captions.

import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
SEED = 42; np.random.seed(SEED)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"; TAB = ROOT / "results" / "tables"; FIG = ROOT / "results" / "figures"

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 120, "pdf.fonttype": 42,
                     "ps.fonttype": 42, "font.size": 11, "axes.edgecolor": "0.2",
                     "axes.linewidth": 0.8, "grid.color": "0.85"})
def savefig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight"); plt.close(fig)

def net_benefit(y, p, pt):
    pred = (p >= pt).astype(int)
    tp = ((pred == 1) & (y == 1)).sum(); fp = ((pred == 1) & (y == 0)).sum()
    n = len(y)
    return tp / n - (fp / n) * (pt / (1 - pt))

# ---- Decision Curve Analysis ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
dca_rows = []; ranges = {}
for ax, sx in zip(axes, ["Male", "Female"]):
    s = pd.read_parquet(DATA / f"scored_{sx.lower()}.parquet")
    y = s["diabetes"].values; p = s["prob_cal"].values
    pts = np.linspace(0.01, 0.6, 60)
    nb_model = np.array([net_benefit(y, p, t) for t in pts])
    prev = y.mean()
    nb_all = np.array([prev - (1 - prev) * (t / (1 - t)) for t in pts])
    ax.plot(pts, nb_model, color="#3b3b3b", lw=1.6, label="XGBoost")
    ax.plot(pts, nb_all, color="#8c8c8c", ls="--", lw=1.2, label="Treat all")
    ax.axhline(0, color="0.4", ls=":", lw=1, label="Treat none")
    ax.set_xlabel("Threshold probability"); ax.set_ylabel("Net benefit")
    ax.text(0.04, 0.06, "(A) Male" if sx == "Male" else "(B) Female",
            transform=ax.transAxes, fontsize=10, fontweight="bold")
    ax.legend(frameon=False, loc="upper right")
    pos = pts[(nb_model > nb_all) & (nb_model > 0)]
    ranges[sx] = (round(pos.min(), 2), round(pos.max(), 2)) if len(pos) else (None, None)
    for t in pts:
        dca_rows.append([sx, round(t, 3), net_benefit(y, p, t)])
savefig(fig, "fig_decision_curve_male_female")
pd.DataFrame(dca_rows, columns=["Sex", "pt", "net_benefit"]).to_csv(TAB / "table_dca.csv", index=False)
print("DCA net-benefit range (model > treat-all & > 0):", ranges)

# ---- Lift charts + high-risk cohort ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
lift_summary = []
for ax, sx in zip(axes, ["Male", "Female"]):
    s = pd.read_parquet(DATA / f"scored_{sx.lower()}.parquet").sort_values("prob_cal", ascending=False).reset_index(drop=True)
    y = s["diabetes"].values; prev = y.mean(); n = len(s)
    deciles = np.arange(1, 11)
    cum_lift = []
    for d in deciles:
        k = int(np.ceil(n * d / 10))
        cum_lift.append((y[:k].mean() / prev))
    bars = ax.bar(deciles * 10, cum_lift, width=7, edgecolor="black", linewidth=0.7)
    # high-risk deciles (cum lift >= 2) shaded darker
    for b, cl in zip(bars, cum_lift):
        b.set_color("#3b3b3b" if cl >= 2 else "#bfbfbf")
    ax.axhline(2, color="0.3", ls="--", lw=1)
    ax.set_xlabel("Top cumulative % of risk distribution"); ax.set_ylabel("Cumulative lift")
    ax.text(0.04, 0.92, "(A) Male" if sx == "Male" else "(B) Female",
            transform=ax.transAxes, fontsize=10, fontweight="bold")
    # operational high-risk boundary = largest top-fraction with cumulative lift >= 2
    frac_grid = np.arange(0.05, 1.0001, 0.05)
    boundary = None
    for fr in frac_grid:
        k = int(np.ceil(n * fr))
        if (y[:k].mean() / prev) >= 2:
            boundary = fr
    if boundary is None:
        boundary = 0.40
    k = int(np.ceil(n * boundary))
    thr_prob = s["prob_cal"].iloc[k - 1]
    hr = s.iloc[:k].copy()
    hr_nd = hr[hr["diabetes"] == 0].copy()
    hr_nd.to_parquet(DATA / f"highrisk_nondiab_{sx.lower()}.parquet", index=False)
    lift_summary.append([sx, n, round(boundary * 100), k, round(y[:k].mean()/prev, 3),
                         round(thr_prob, 3), len(hr), len(hr_nd)])
savefig(fig, "fig_lift_male_female")

ls = pd.DataFrame(lift_summary, columns=["Sex", "N_total", "HighRisk_topPct", "HighRisk_n",
                                         "Cum_lift", "Prob_threshold", "HighRisk_all", "HighRisk_nondiabetic"])
ls.to_csv(TAB / "table_lift_highrisk.csv", index=False)
print("\nLift-based high-risk cohort:")
print(ls.to_string(index=False))
print("\nSaved DCA/lift figures, tables, and non-diabetic high-risk cohorts.")


DCA net-benefit range (model > treat-all & > 0): {'Male': (np.float64(0.01), np.float64(0.57)), 'Female': (np.float64(0.01), np.float64(0.48))}



Lift-based high-risk cohort:
   Sex  N_total  HighRisk_topPct  HighRisk_n  Cum_lift  Prob_threshold  HighRisk_all  HighRisk_nondiabetic
  Male     9999               25        2500     2.043           0.303          2500                  1527
Female    13105               40        5242     2.056           0.139          5242                  3787

Saved DCA/lift figures, tables, and non-diabetic high-risk cohorts.
